In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import Dataset
import pandas as pd

# 1. Configuración del modelo
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 2. Agregar adaptadores LoRA para el entrenamiento eficiente
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

# 3. Preparación del Dataset

data = []
with open("/content/drive/MyDrive/DatosIberLEF/train/MSLG_SPA_train.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()[1:]
    for line in lines:
        parts = line.strip().split("\t")
        if len(parts) == 3:
            data.append({"glosa": parts[1], "espanol": parts[2]})

dataset = Dataset.from_list(data)

prompt_style = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Eres un traductor experto de Lengua de Señas Mexicana (LSM). Traduce la glosa proporcionada a un español natural y fluido.<|eot_id|><|start_header_id|>user<|end_header_id|>
Glosa: {}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Traducción: {}<|eot_id|>"""

def formatting_prompts_func(examples):
    inputs       = examples["glosa"]
    outputs      = examples["espanol"]
    texts = []
    for input_text, output_text in zip(inputs, outputs):
        t = prompt_style.format(input_text, output_text)
        texts.append(t)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

# 4. Entrenamiento
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 15,
        max_steps = 250,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

trainer.train()



==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Map:   0%|          | 0/490 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/490 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 490 | Num Epochs = 5 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,4.210525
2,4.181026
3,4.067099
4,3.879502
5,3.871272
6,3.463593
7,3.374497
8,2.957949
9,2.932455
10,2.653415


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-250/tokenizer_config.json.


TrainOutput(global_step=250, training_loss=0.692964309155941, metrics={'train_runtime': 364.7954, 'train_samples_per_second': 5.483, 'train_steps_per_second': 0.685, 'total_flos': 2976923943800832.0, 'train_loss': 0.692964309155941, 'epoch': 4.03265306122449})

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Cargar la base de datos original
# Leemos el archivo respetando las tabulaciones (\t) como separadores
df = pd.read_csv("/content/drive/MyDrive/DatosIberLEF/train/MSLG_SPA_train.txt", sep="\t", header=0)

# 2. Dividir los datos: 90% para entrenamiento, 10% para validación
train_df, val_df = train_test_split(df, test_size=0.10, random_state=42)

# 3. Guardar los nuevos conjuntos en archivos de texto separados
train_df.to_csv("train_90.txt", sep="\t", index=False)
val_df.to_csv("val_10.txt", sep="\t", index=False)

# 4. Mostrar el resumen de la operación
print("--- RESUMEN DE LA DIVISIÓN DE DATOS ---")
print(f"Total de pares originales: {len(df)}")
print(f"Pares para ENTRENAMIENTO (90%): {len(train_df)} (Guardado como 'train_90.txt')")
print(f"Pares para VALIDACIÓN (10%): {len(val_df)} (Guardado como 'val_10.txt')")

--- RESUMEN DE LA DIVISIÓN DE DATOS ---
Total de pares originales: 490
Pares para ENTRENAMIENTO (90%): 441 (Guardado como 'train_90.txt')
Pares para VALIDACIÓN (10%): 49 (Guardado como 'val_10.txt')


# Aqui se realiza la prueba interna con el split del 10% de los datos


In [ ]:

!pip install evaluate sacrebleu tqdm

import pandas as pd
import torch
from unsloth import FastLanguageModel
import evaluate
from tqdm import tqdm # Para ver una barra de progreso

metric_bleu = evaluate.load("sacrebleu")
metric_chrf = evaluate.load("chrf")

val_df = pd.read_csv("val_10.txt", sep="\t")
glosas_val = val_df["MSLG"].tolist()
referencias_espanol = val_df["SPA"].tolist()

FastLanguageModel.for_inference(model)

inference_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Eres un traductor experto de Lengua de Señas Mexicana (LSM). Traduce la glosa proporcionada a un español natural y fluido.<|eot_id|><|start_header_id|>user<|end_header_id|>
Glosa: {}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Traducción: """

predicciones = []

# 5. Para generar traducciones una por una

print("Traduciendo el conjunto de validación...")
for glosa in tqdm(glosas_val):
    prompt = inference_prompt.format(glosa)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=64,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.1,
            do_sample=False
        )

    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]


    if "Traducción:" in decoded_output:
        prediccion = decoded_output.split("Traducción:")[-1].strip()
    else:
        prediccion = decoded_output.strip()

    predicciones.append(prediccion)

# 6. Calcular las métricas

referencias_formateadas = [[ref] for ref in referencias_espanol]

resultados_bleu = metric_bleu.compute(predictions=predicciones, references=referencias_formateadas)
resultados_chrf = metric_chrf.compute(predictions=predicciones, references=referencias_formateadas, word_order=2)

# 7. Imprimir el reporte
print("\n" + "="*50)
print("   RESULTADOS DE EVALUACIÓN (VALIDACIÓN INTERNA)   ")
print("="*50)
print(f"Total de frases evaluadas: {len(predicciones)}")
print(f"Puntaje BLEU:   {round(resultados_bleu['score'], 2)}")
print(f"Puntaje chrF++: {round(resultados_chrf['score'], 2)}")
print("="*50)

# 8. Mostrar una pequeña muestra para inspección visual
print("\n--- MUESTRA DE TRADUCCIONES ---")
for i in range(min(3, len(predicciones))):
    print(f"Glosa original: {glosas_val[i]}")
    print(f"Referencia:     {referencias_espanol[i]}")
    print(f"Tu Modelo:      {predicciones[i]}")
    print("-" * 50)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.7 MB/s eta 0:00:00


Traduciendo el conjunto de validación...


  0%|          | 0/49 [00:00<?, ?it/s]Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, Fu


   RESULTADOS DE EVALUACIÓN (VALIDACIÓN INTERNA)   
Total de frases evaluadas: 49
Puntaje BLEU:   77.65
Puntaje chrF++: 87.11

--- MUESTRA DE TRADUCCIONES ---
Glosa original: JABÓN BURBUJAS
Referencia:     El jabón hace burbujas.
Tu Modelo:      El jabón hace burbujas.
--------------------------------------------------
Glosa original: APAGAR LUZ POR-FAVOR
Referencia:     Por favor, apaga la luz.
Tu Modelo:      Apaga la luz, por favor.
--------------------------------------------------
Glosa original: TÚ NO PREOCUPAR DEJARLO-ASÍ
Referencia:     No te preocupes, déjalo así.
Tu Modelo:      No te preocupes, déjalo así.
--------------------------------------------------


In [ ]:
import torch

# 1. Ver los parámetros de los adaptadores LoRA
print("--- RESUMEN DE PARÁMETROS ---")
model.print_trainable_parameters()

# 2. Calcular el uso de memoria VRAM
gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
reserved_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

print(f"\n--- HUELLA DE MEMORIA GPU ---")
print(f"GPU Activa: {gpu_stats.name}")
print(f"Capacidad Total VRAM: {max_memory} GB")
print(f"VRAM reservada por el modelo: {reserved_memory} GB")
print(f"Porcentaje de VRAM utilizada: {round((reserved_memory / max_memory) * 100, 2)}%")

--- RESUMEN DE PARÁMETROS ---
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511

--- HUELLA DE MEMORIA GPU ---
GPU Activa: Tesla T4
Capacidad Total VRAM: 14.563 GB
VRAM reservada por el modelo: 3.449 GB
Porcentaje de VRAM utilizada: 23.68%


In [ ]:
# 1. Fusionar y guardar el modelo SPA -> MSLG
print("Guardar modelo GLosa a ESP")
model.save_pretrained_merged(
    "Modelo_IberLEF_LSM2ESP",
    tokenizer,
    save_method = "merged_16bit"
)


Guardar modelo GLosa a ESP


Unsloth: Restored added_tokens_decoder metadata in Modelo_IberLEF_LSM2ESP/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [03:32<00:00, 106.28s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:13<00:00, 96.76s/it]


Unsloth: Merge process complete. Saved to `/content/Modelo_IberLEF_LSM2ESP`


# En esta parte es importante señalar que el modelo se guarda en Drive, hay que modificar la ruta si se requiere, en caso de su replicacion local esta celda es inecesaria.

In [ ]:
import shutil
import os

# Definimos las rutas
origen = '/content/Modelo_IberLEF_LSM2ESP'
destino = '/content/drive/MyDrive/DatosIberLEF'


if os.path.exists(origen):

    shutil.copytree(origen, destino, dirs_exist_ok=True)
    print(f"✅ ¡Listo! El modelo se ha copiado con éxito en: {destino}")
else:
    print("❌ Error: No se encontró la carpeta de origen. Revisa la ruta.")

✅ ¡Listo! El modelo se ha copiado con éxito en: /content/drive/MyDrive/DatosIberLEF


In [ ]:
import shutil
import os
from google.colab import drive

drive.mount('/content/drive')

origen = '/content/Modelo_IberLEF_LSM2ESP'

destino = '/content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_LSM2ESP'

if os.path.exists(origen):
    shutil.copytree(origen, destino, dirs_exist_ok=True)
    print(f"✅ Carpeta copiada íntegramente en: {destino}")
else:
    print("❌ La carpeta de origen no existe.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Carpeta copiada íntegramente en: /content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_LSM2ESP


# RUTA INVERSA - Las siguientes celdas se utilizaran para la traduccion del Español al LSM

In [ ]:
import pandas as pd
from datasets import Dataset

# Rutas definidas
path_train = "/content/drive/MyDrive/DatosIberLEF/train/MSLG_SPA_train.txt"

# Cargar y preparar el dataset
df_train = pd.read_csv(path_train, sep="\t")

# Prompt para la ruta inversa
prompt_style_spa2mslg = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Eres un traductor experto de español a Lengua de Señas Mexicana (LSM). Traduce el texto en español proporcionado a su representación exacta en glosa, respetando estrictamente las convenciones de anotación (guiones, signos de suma, prefijos dm- y #). No utilices tildes ni acentos ortográficos en las glosas generadas.<|eot_id|><|start_header_id|>user<|end_header_id|>
Español: {}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Glosa: {}<|eot_id|>"""

def formatting_prompts_spa2mslg(examples):
    inputs  = examples["SPA"]
    outputs = examples["MSLG"]
    texts = []
    for input_text, output_text in zip(inputs, outputs):
        t = prompt_style_spa2mslg.format(input_text, output_text)
        texts.append(t)
    return { "text" : texts }

# Crear el dataset de Hugging Face
dataset_inverso = Dataset.from_pandas(df_train)
dataset_inverso = dataset_inverso.map(formatting_prompts_spa2mslg, batched=True)

print(f"Dataset inverso preparado con {len(dataset_inverso)} ejemplos.")

Map:   0%|          | 0/490 [00:00<?, ? examples/s]

Dataset inverso preparado con 490 ejemplos.


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

# Cargar modelo base limpio
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Agregar adaptadores LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

# Entrenar
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_inverso,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 15,
        max_steps = 300,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        output_dir = "outputs_spa2mslg",
        optim = "adamw_8bit",
    ),
)

trainer.train()

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/490 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 490 | Num Epochs = 5 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
10,3.206554
20,1.351575
30,0.572035
40,0.543112
50,0.507352
60,0.480742
70,0.360472
80,0.327735
90,0.316189
100,0.285502


Unsloth: Restored added_tokens_decoder metadata in outputs_spa2mslg/checkpoint-300/tokenizer_config.json.


TrainOutput(global_step=300, training_loss=0.3979733645915985, metrics={'train_runtime': 562.4943, 'train_samples_per_second': 4.267, 'train_steps_per_second': 0.533, 'total_flos': 5930204990263296.0, 'train_loss': 0.3979733645915985, 'epoch': 4.848979591836734})

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt_style = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Eres un traductor experto de español a Lengua de Señas Mexicana (LSM). Traduce el texto en español proporcionado a su representación exacta en glosa, respetando estrictamente las convenciones de anotación (guiones, signos de suma, prefijos dm- y #). No utilices tildes ni acentos ortográficos en las glosas generadas.<|eot_id|><|start_header_id|>user<|end_header_id|>
Español: {}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Glosa: """

frase_test = "Ayer llegó mi tío de San Francisco."
prompt = test_prompt_style.format(frase_test)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=False)

resultado = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(f"Entrada: {frase_test}")
print(f"Salida del modelo: {resultado.split('Glosa:')[-1].strip()}")

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Entrada: Ayer llegó mi tío de San Francisco.
Salida del modelo: AYER MI TIO SAN FRANCISCO YA LLEGAR


In [ ]:
# Guardar versión fusionada localmente primero
model.save_pretrained_merged(
    "Modelo_IberLEF_SPA2MSLG_Local",
    tokenizer,
    save_method = "merged_16bit"
)
print("Modelo fusionado localmente.")

Unsloth: Restored added_tokens_decoder metadata in Modelo_IberLEF_SPA2MSLG_Local/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [03:38<00:00, 109.18s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:30<00:00, 45.03s/it]


Unsloth: Merge process complete. Saved to `/content/Modelo_IberLEF_SPA2MSLG_Local`
Modelo fusionado localmente.


# En esta parte es importante señalar que el modelo se guarda en Drive, hay que modificar la ruta si se requiere, en caso de su replicacion local esta celda es inecesaria.

In [ ]:
import shutil
import os

# Ruta destino en Drive
drive_dest = "/content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_SPA2MSLG"

# Crear carpeta si no existe
if not os.path.exists(drive_dest):
    os.makedirs(drive_dest)

# Copiar contenido de la carpeta local al Drive
# Nota: Usamos un comando de sistema para mayor rapidez en carpetas grandes
!cp -r Modelo_IberLEF_SPA2MSLG_Local/* "{drive_dest}"

print(f"Modelo copiado exitosamente a: {drive_dest}")

Modelo copiado exitosamente a: /content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_SPA2MSLG


# Aqui se realiza la prueba a ciegas de donde salen los resultados finales



In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
import unicodedata
from unsloth import FastLanguageModel
from tqdm import tqdm

# --- 1. CONFIGURACIÓN DE RUTAS Y NOMBRES ---
# Rutas en tu Drive
model_path = "/content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_SPA2MSLG"
test_file_path = "/content/drive/MyDrive/DatosIberLEF/test/SPA2MSLG_test.txt"

# Nomenclatura oficial: TeamName_SolutionName_Subtask.txt
team_name = "LSM-INAOE"
solution_name = "Llama3.2"
output_file_path = f"/content/drive/MyDrive/DatosIberLEF/{team_name}_{solution_name}_SPA2MSLG.txt"

# --- 2. CARGA DEL MODELO ---
print("Cargando modelo SPA2MSLG desde Drive...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# --- 3. CARGA DE DATOS DE PRUEBA ---
print("Cargando datos de prueba oficiales...")
df_test = pd.read_csv(test_file_path, sep="\t", header=0)
ids = df_test["ID"].tolist()
frases_espanol = df_test["SPA"].tolist()

# --- 4. PREPARACIÓN DE INFERENCIA ---
inference_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Eres un traductor experto de español a Lengua de Señas Mexicana (LSM). Traduce el texto en español proporcionado a su representación exacta en glosa, respetando estrictamente las convenciones de anotación (guiones, signos de suma, prefijos dm- y #). No utilices tildes ni acentos ortográficos en las glosas generadas.<|eot_id|><|start_header_id|>user<|end_header_id|>
Español: {}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Glosa: """

def quitar_tildes(texto):
    """Elimina acentos ortográficos respetando caracteres especiales como #, +, -, dm-"""
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

predicciones = []

# --- 5. GENERACIÓN DE GLOSAS ---
print(f"Traduciendo {len(frases_espanol)} frases del conjunto de prueba (SPA -> MSLG)...")
for frase in tqdm(frases_espanol):
    prompt = inference_prompt.format(frase)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=64,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.1,
            do_sample=False
        )

    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Extraer la respuesta
    if "Glosa:" in decoded_output:
        prediccion = decoded_output.split("Glosa:")[-1].strip()
    else:
        prediccion = decoded_output.strip()


    prediccion = prediccion.replace('\n', ' ').replace('\r', '')
    prediccion = quitar_tildes(prediccion).upper()
    prediccion = prediccion.replace('DM-', 'dm-')
    predicciones.append(prediccion)

# --- 6. ESCRITURA DEL ARCHIVO FINAL ---
print(f"\nEscribiendo resultados en {output_file_path}...")
with open(output_file_path, "w", encoding="utf-8", newline='\n') as f:
    for id_val, pred in zip(ids, predicciones):
        linea = f'"{id_val}"\t"{pred}"\n'
        f.write(linea)

print(f"\n¡Archivo de entrega generado con éxito! Revisa la ruta: {output_file_path}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Cargando modelo SPA2MSLG desde Drive...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_SPA2MSLG' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_SPA2MSLG' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_SPA2MSLG as a legacy tokenizer.


Cargando datos de prueba oficiales...
Traduciendo 245 frases del conjunto de prueba (SPA -> MSLG)...


  0%|          | 0/245 [00:00<?, ?it/s]Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, F


Escribiendo resultados en /content/drive/MyDrive/DatosIberLEF/LSM-INAOE_Llama3.2_SPA2MSLG.txt...

¡Archivo de entrega generado con éxito! Revisa la ruta: /content/drive/MyDrive/DatosIberLEF/LSM-INAOE_Llama3.2_SPA2MSLG.txt


In [ ]:
# --- 1. IMPORTACIONES ---
import pandas as pd
import torch
from unsloth import FastLanguageModel
from tqdm import tqdm
import gc

# Limpieza de memoria por seguridad
torch.cuda.empty_cache()
gc.collect()

# --- 2. CONFIGURACIÓN DE RUTAS Y NOMBRES ---
model_path = "/content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_LSM2ESP"
test_file_path = "/content/drive/MyDrive/DatosIberLEF/test/MSLG2SPA_test.txt"

# Nomenclatura oficial
team_name = "LSM-INAOE"
solution_name = "Llama3.2"
output_file_path = f"/content/drive/MyDrive/DatosIberLEF/{team_name}_{solution_name}_MSLG2SPA.txt"

# --- 3. CARGA DEL MODELO ---
print("Cargando modelo MSLG2SPA desde Drive...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# --- 4. CARGA DE DATOS DE PRUEBA ---
print("Cargando datos de prueba oficiales...")
df_test = pd.read_csv(test_file_path, sep="\t", header=0)
ids = df_test["ID"].tolist()
glosas = df_test["MSLG"].tolist()

# --- 5. PREPARACIÓN DE INFERENCIA ---
inference_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Eres un traductor experto de Lengua de Señas Mexicana (LSM). Traduce la glosa proporcionada a un español natural y fluido.<|eot_id|><|start_header_id|>user<|end_header_id|>
Glosa: {}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Traducción: """

predicciones = []

# --- 6. GENERACIÓN DE FRASES EN EESPAÑOL ---
print(f"Traduciendo {len(glosas)} glosas del conjunto de prueba (MSLG -> SPA)...")
for glosa in tqdm(glosas):
    prompt = inference_prompt.format(glosa)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=64,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.1,
            do_sample=False
        )

    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Extraer la traducción
    if "Traducción:" in decoded_output:
        prediccion = decoded_output.split("Traducción:")[-1].strip()
    else:
        prediccion = decoded_output.strip()

    prediccion = prediccion.replace('\n', ' ').replace('\r', '')

    predicciones.append(prediccion)

# --- 7. ESCRITURA DEL ARCHIVO FINAL ---
print(f"\nEscribiendo resultados en {output_file_path}...")
with open(output_file_path, "w", encoding="utf-8", newline='\n') as f:
    for id_val, pred in zip(ids, predicciones):
        linea = f'"{id_val}"\t"{pred}"\n'
        f.write(linea)

print(f"\n¡Archivo de entrega generado con éxito! Revisa la ruta: {output_file_path}")

Cargando modelo MSLG2SPA desde Drive...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_LSM2ESP' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_LSM2ESP' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/drive/MyDrive/DatosIberLEF/Modelo_IberLEF_LSM2ESP as a legacy tokenizer.


Cargando datos de prueba oficiales...
Traduciendo 245 glosas del conjunto de prueba (MSLG -> SPA)...


  0%|          | 0/245 [00:00<?, ?it/s]Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, F


Escribiendo resultados en /content/drive/MyDrive/DatosIberLEF/LSM-INAOE_Llama3.2_MSLG2SPA.txt...

¡Archivo de entrega generado con éxito! Revisa la ruta: /content/drive/MyDrive/DatosIberLEF/LSM-INAOE_Llama3.2_MSLG2SPA.txt


# Estudio de ablacion

In [1]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes evaluate sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.5 MB/s eta 0:00:00


In [2]:
!pip install evaluate sacrebleu

In [5]:
import pandas as pd
import torch
from unsloth import FastLanguageModel
import evaluate
from tqdm import tqdm

# Cargar métricas
metric_bleu = evaluate.load("sacrebleu")
metric_chrf = evaluate.load("chrf")

# Cargar datos de validación
val_df = pd.read_csv("val_10.txt", sep="\t")
glosas_val = val_df["MSLG"].tolist()
referencias_espanol = val_df["SPA"].tolist()

# 1. CARGAR MODELO BASE (Sin LoRA)
print("Cargando modelo Base...")
model_base, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_base)

predicciones_base = []
predicciones_prompt = []

# Definir el prompt experto (Escenario 2)
inference_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Eres un traductor experto de Lengua de Señas Mexicana (LSM). Traduce la glosa proporcionada a un español natural y fluido.<|eot_id|><|start_header_id|>user<|end_header_id|>
Glosa: {}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Traducción: """

print("Traduciendo...")
for glosa in tqdm(glosas_val):
    # --- Escenario 1: Base Llama  ---
    inputs_base = tokenizer([f"Traduce a español: {glosa}"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        out_base = model_base.generate(**inputs_base, max_new_tokens=64, use_cache=False, temperature=0.1, do_sample=False)
    pred_base = tokenizer.batch_decode(out_base, skip_special_tokens=True)[0].split("español:")[-1].strip()
    predicciones_base.append(pred_base.replace('\n', ' '))

    # --- Escenario 2: Prompt-only  ---
    inputs_prompt = tokenizer([inference_prompt.format(glosa)], return_tensors="pt").to("cuda")
    with torch.no_grad():
        out_prompt = model_base.generate(**inputs_prompt, max_new_tokens=64, use_cache=False, temperature=0.1, do_sample=False)
    pred_prompt = tokenizer.batch_decode(out_prompt, skip_special_tokens=True)[0]
    pred_prompt = pred_prompt.split("Traducción:")[-1].strip() if "Traducción:" in pred_prompt else pred_prompt.strip()
    predicciones_prompt.append(pred_prompt.replace('\n', ' '))

# Evaluar
refs = [[ref] for ref in referencias_espanol]

bleu_base = metric_bleu.compute(predictions=predicciones_base, references=refs)['score']
chrf_base = metric_chrf.compute(predictions=predicciones_base, references=refs, word_order=2)['score']

bleu_prompt = metric_bleu.compute(predictions=predicciones_prompt, references=refs)['score']
chrf_prompt = metric_chrf.compute(predictions=predicciones_prompt, references=refs, word_order=2)['score']

print(f"--- RESULTADOS DEL ABLATION STUDY ---")
print(f"Base Llama -> BLEU: {bleu_base:.2f} | chrF++: {chrf_base:.2f}")
print(f"Prompt-only -> BLEU: {bleu_prompt:.2f} | chrF++: {chrf_prompt:.2f}")

Cargando modelo Base...
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Traduciendo...


  0%|          | 0/49 [00:00<?, ?it/s]Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, Fu

--- RESULTADOS DEL ABLATION STUDY ---
Base Llama -> BLEU: 0.29 | chrF++: 11.94
Prompt-only -> BLEU: 5.38 | chrF++: 26.37
